# Process your CSVs to get a final data output

Fill in your folder path below and then Run All to get graphs of your data and a summary csv file

In [ ]:
# Path to the folder with your csv files
path = r''
save_traces = True

## More filtering ideas:
- Constrain head and tailbase to be within a certaindistance of each other, otherwise throw out frames
- Need to figure out what is happening in the videos for these big jumps


Preprocessing by removing low likelihood points and outliers

In [ ]:
import pandas as pd
import numpy as np

def clip_start_by_last_bad_run(df, part, fps=10, start_clip_sec=60, bad_like_thresh=0):
    """
    Clip the start of the video based on the last run of consecutive frames
    within the first `start_clip_sec` seconds where likelihood < bad_like_thresh.
    """
    like_col = f"{part}_like"
    max_frames = int(start_clip_sec * fps)
    likes = df[like_col].to_numpy(dtype=float)
    likes_to_check = likes[:max_frames] if len(likes) > max_frames else likes

    last_run_end = None
    for i, val in enumerate(likes_to_check):
        if val < bad_like_thresh:
            last_run_end = i

    if last_run_end is not None:
        df = df.iloc[last_run_end + 1:].reset_index(drop=True)

    return df


def replace_low_likelihood(df, part, like_thresh=0.7):
    """
    Replace X/Y coords where likelihood < like_thresh with linear interpolation.
    """
    x_col, y_col, like_col = f"{part}_x", f"{part}_y", f"{part}_like"
    coords = df[[x_col, y_col]].to_numpy(dtype=float)
    likes = df[like_col].to_numpy(dtype=float)

    replaced_count = 0
    i = 0
    while i < len(df):
        if likes[i] < like_thresh:
            start = i - 1
            while start >= 0 and likes[start] < like_thresh:
                start -= 1
            end = i + 1
            while end < len(df) and likes[end] < like_thresh:
                end += 1
            if start >= 0 and end < len(df):
                n_points = end - start - 1
                x_start, y_start = coords[start]
                x_end, y_end = coords[end]
                for j in range(1, n_points+1):
                    alpha = j / (n_points+1)
                    coords[start+j, 0] = (1-alpha)*x_start + alpha*x_end
                    coords[start+j, 1] = (1-alpha)*y_start + alpha*y_end
                    replaced_count += 1
            i = end
        else:
            i += 1

    df[x_col] = coords[:, 0]
    df[y_col] = coords[:, 1]
    return df, replaced_count


def clip_outliers_by_std(df, part, n_std=1.5):
    """
    Replace coords (X/Y) outside N std of the mean with NaN, then interpolate.
    """
    x_col, y_col = f"{part}_x", f"{part}_y"
    x = df[x_col].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)

    mean_x, std_x = np.nanmean(x), np.nanstd(x)
    mean_y, std_y = np.nanmean(y), np.nanstd(y)

    mask = ((np.abs(x - mean_x) > n_std * std_x) |
            (np.abs(y - mean_y) > n_std * std_y))

    x[mask] = np.nan
    y[mask] = np.nan

    df[x_col] = pd.Series(x).interpolate(method="linear").bfill().ffill()
    df[y_col] = pd.Series(y).interpolate(method="linear").bfill().ffill()

    return df, mask.sum()


def preprocess_csv(file_path, part="TailBase", fps=10, start_clip_sec=60,
                   bad_like_thresh=0, like_thresh=0.7, n_std=3,
                   verbose=True):
    """
    Preprocess a single DeepLabCut-style CSV file:
      1. Clip start based on low-likelihood runs
      2. Replace low-likelihood coords
      3. Remove outliers by std threshold

    Returns: (df_raw, df_clean, summary_dict)
    """
    if verbose:
        print("Using file:", file_path)

    # --- Load CSV ---
    df_raw = pd.read_csv(file_path, header=[1])
    df_raw = df_raw.drop(index=0).reset_index(drop=True)
    df_raw.columns = ["Frame", "Head_x", "Head_y", "Head_like",
                      "TailBase_x", "TailBase_y", "TailBase_like"]

    for col in ["Frame", "Head_x", "Head_y", "TailBase_x", "TailBase_y",
                "Head_like", "TailBase_like"]:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

    df_raw = df_raw.dropna(subset=["Frame", f"{part}_y"])
    df_raw["Frame"] = df_raw["Frame"].astype(int)

    # --- Step 1: clip start ---
    df_clip = clip_start_by_last_bad_run(df_raw.copy(), part,
                                         fps=fps,
                                         start_clip_sec=start_clip_sec,
                                         bad_like_thresh=bad_like_thresh)
    if verbose:
        print(f"{len(df_raw) - len(df_clip)} frames removed from start.")

    # --- Step 2: replace low likelihood ---
    df_like, n_low = replace_low_likelihood(df_clip.copy(), part,
                                            like_thresh=like_thresh)
    if verbose:
        print(f"Replaced {n_low} low-likelihood points (< {like_thresh}).")

    # --- Step 3: clip outliers ---
    df_std, n_out = clip_outliers_by_std(df_like.copy(), part, n_std=n_std)
    if verbose:
        print(f"Removed {n_out} outlier points (> {n_std} std).")

    summary = {
        "frames_removed": len(df_raw) - len(df_clip),
        "low_points_replaced": n_low,
        "outliers_removed": n_out
    }

    return df_raw, df_std, summary

Now plotting tailbase trajectories and estimating the circle path

## This saves the trajectories

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import matplotlib.cm as cm
import os
import glob
import pandas as pd
import numpy as np
from scipy.optimize import leastsq

plt.style.use('dark_background')

folder_path = path
fps = 10

# --- circle fit helper ---
def fit_circle(x, y):
    # initial guess at mean
    x_m, y_m = np.mean(x), np.mean(y)
    
    def calc_R(xc, yc):
        return np.sqrt((x - xc)**2 + (y - yc)**2)

    def f(c):
        Ri = calc_R(*c)
        return Ri - Ri.mean()

    center_estimate = (x_m, y_m)
    center, _ = leastsq(f, center_estimate)
    xc, yc = center
    R = calc_R(xc, yc).mean()
    return xc, yc, R


# --- get all CSVs ---
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {folder_path}")

output_folder = os.path.join(folder_path, "Traces")
os.makedirs(output_folder, exist_ok=True)

for file_path in csv_files:
    filename = os.path.basename(file_path)
    animal_id = filename[:5]

    # -------- Preprocess with our function --------
    df_raw, df, summary = preprocess_csv(file_path, part="TailBase", fps=fps)
    # print(f"{animal_id} - Preprocessing summary:", summary)

    # Extract arrays
    x = df["TailBase_x"].to_numpy()
    y = df["TailBase_y"].to_numpy()
    frames = df["Frame"].to_numpy()

    # ---- fit circle to cleaned trajectory ----
    xc, yc, R = fit_circle(x, y)

    # ---- Colored trajectory ----
    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    # colormap by frame number
    norm = plt.Normalize(frames.min(), frames.max())
    lc = LineCollection(segments, cmap='plasma', norm=norm)
    lc.set_array(frames)
    lc.set_linewidth(2)

    plt.figure(figsize=(8, 8))
    ax = plt.gca()

    # add trajectory
    line = ax.add_collection(lc)
    plt.colorbar(line, ax=ax, label="Frame (time)")

    # add fitted circle + center
    ax.scatter(xc, yc, color="white", s=200, marker="x", label="Estimated center", linewidth=3, zorder=4)
    circle = plt.Circle((xc, yc), R, color="white", fill=False, linestyle="--", alpha=1, linewidth=3, zorder=4)
    ax.add_patch(circle)

    ax.set_xlabel("TailBase X")
    ax.set_ylabel("TailBase Y")
    ax.set_title(f"TailBase trajectory - {animal_id}")
    ax.set_aspect("equal", adjustable="box")
    ax.legend()

    if save_traces:
        input_name = os.path.splitext(filename)[0]
        output_path = os.path.join(output_folder, f"{input_name}_TRACE.png")

        plt.savefig(output_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved: {output_path}")
        
    else:
        plt.show()